In [69]:
import bayespy as bp
import numpy as np
import csv

# -------------------------------
# LOAD DATA
# -------------------------------
data = []
with open("data.csv") as f:
    reader = csv.reader(f)
    for row in reader:
        data.append(row)

data = np.array(data)   # keep as string initially
N = len(data)

# -------------------------------
# HEADERS
# -------------------------------
headers = [
    'age', 'sex', 'family_history',
    'diet', 'sportiness', 'cholesterol',
    'has_heart_disease'
]

# -------------------------------
# CREATE ENUM MAPPING
# -------------------------------
mapping = {}

for col in range(data.shape[1]):
    column_name = headers[col]
    unique_values = np.unique(data[:, col])

    mapping[column_name] = {}
    for i, val in enumerate(unique_values):
        mapping[column_name][val] = i

# print mapping
print("\nMappings:")
for k, v in mapping.items():
    print(k, ":", v)

# -------------------------------
# CONVERT DATA → NUMERIC
# -------------------------------
numeric_data = np.zeros_like(data, dtype=int)

for col in range(data.shape[1]):
    column_name = headers[col]
    for row in range(N):
        numeric_data[row][col] = mapping[column_name][data[row][col]]
print(numeric_data)
# -------------------------------
# CREATE NODE FUNCTION
# -------------------------------
def create_node(col):
    uniq_values_count = len(mapping[headers[col]])   # number of categories

    p = bp.nodes.Dirichlet(np.ones(uniq_values_count))
    node = bp.nodes.Categorical(p, plates=(N,))

    node.observe(numeric_data[:, col])   # observe numeric values
    return node

# -------------------------------
# CREATE FEATURE NODES
# -------------------------------
age = create_node(0)
sex = create_node(1)
family = create_node(2)
diet = create_node(3)
sportiness = create_node(4)
cholesterol = create_node(5)

# -------------------------------
# HEART DISEASE NODE
# -------------------------------
p_hd = bp.nodes.Dirichlet(
    np.ones(2),# yes  or no
    plates=(
        len(mapping['age']),
        len(mapping['sex']),
        len(mapping['family_history']),
        len(mapping['diet']),
        len(mapping['sportiness']),
        len(mapping['cholesterol'])
    )
)

heart = bp.nodes.MultiMixture(
    [age, sex, family, diet, sportiness, cholesterol],
    bp.nodes.Categorical,
    p_hd
)

# observe actual output
heart.observe(numeric_data[:, 6])

# train/update
p_hd.update()

# -------------------------------
# PREDICTION LOOP
# -------------------------------
while True:
    print("\nEnter values using mapping:")

    input_vals = []
    for col in range(6):
        print(headers[col], mapping[headers[col]])
        val = int(input("Enter value: "))
        input_vals.append(val)

    prob = bp.nodes.MultiMixture(
        input_vals,
        bp.nodes.Categorical,
        p_hd
    ).get_moments()[0][ mapping['has_heart_disease']['Yes'] ]

    print("\nProbability of Heart Disease:", prob)

    if input("Continue? (y/n): ") == 'n':
        break


Mappings:
age : {np.str_('MiddleAged'): 0, np.str_('SeniorCitizen'): 1, np.str_('SuperSeniorCitizen'): 2, np.str_('Teen'): 3, np.str_('Youth'): 4}
sex : {np.str_('Female'): 0, np.str_('Male'): 1}
family_history : {np.str_('No'): 0, np.str_('Yes'): 1}
diet : {np.str_('High'): 0, np.str_('Low'): 1, np.str_('Medium'): 2}
sportiness : {np.str_('Active'): 0, np.str_('Athlete'): 1, np.str_('Moderate'): 2, np.str_('Sedentary'): 3}
cholesterol : {np.str_('BorderLine'): 0, np.str_('High'): 1, np.str_('Normal'): 2}
has_heart_disease : {np.str_('No'): 0, np.str_('Yes'): 1}
[[4 1 1 0 0 1 1]
 [4 0 0 1 1 2 0]
 [0 1 1 0 2 0 1]
 [1 0 1 2 3 1 1]
 [1 1 0 2 0 2 0]
 [0 0 0 1 2 2 0]
 [2 1 1 0 3 1 1]
 [3 0 0 1 1 2 0]
 [4 1 0 2 0 0 0]
 [0 1 1 0 2 1 1]
 [1 0 1 2 3 0 1]
 [4 0 0 1 0 2 0]
 [2 0 1 0 3 1 1]
 [3 1 0 1 1 2 0]
 [0 0 1 2 2 0 1]
 [1 1 1 0 3 1 1]
 [4 1 0 2 0 2 0]
 [0 1 1 0 2 0 1]
 [1 0 0 1 0 2 0]
 [2 1 1 0 3 1 1]]

Enter values using mapping:
age {np.str_('MiddleAged'): 0, np.str_('SeniorCitizen'): 1, 

Enter value:  sd


ValueError: invalid literal for int() with base 10: 'sd'